# 01 · Bronze — ingesta incremental

Runner delgado (skill `de-developer`): la lógica vive en `src/`. Extrae la ventana `(watermark, fin]` de Open-Meteo y del simulador IoT, agrega linaje y guarda el lote en `storage/bronze/`.

Requisito: PostgreSQL levantado (`docker compose up -d`) y `.env` configurado.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src import pipeline
from src.common import db, lake
from src.config import get_settings, load_pipeline

engine = db.get_engine()
settings, cfg = get_settings(), load_pipeline()

In [ ]:
# Fuentes a ingestar: None = todas las habilitadas en config/pipeline.yaml
SOURCES = None

BATCH_ID = pipeline.run_bronze(engine, sources=SOURCES)
BATCH_ID

In [ ]:
import pandas as pd

bronze = pd.concat([lake.read_layer(settings.storage_path, 'bronze', cfg['domain'], s, BATCH_ID) for s in pipeline.CONNECTORS], ignore_index=True)
print(len(bronze), 'registros')
bronze.head(10)

In [ ]:
pd.read_sql('SELECT * FROM etl.watermarks ORDER BY entity', engine)